# Module 3: Gold Layer DQ - Business Rules Catalog

## Learning Objectives
- Create a centralized Rules Catalog table
- Define business rules as data (not just code)
- Build a stored procedure that auto-provisions DMFs from catalog entries
- Understand the "Rules as Data" pattern for DQ governance

## Key Concept: Rules as Data

Instead of hard-coding DQ checks, we store rules in a **catalog table**. This enables:
- **Self-service**: Data stewards add rules without writing SQL
- **Auditability**: Full history of who added what rule and when
- **Automation**: A procedure reads the catalog and creates DMFs automatically
- **Governance**: Rules have owners, severity levels, and lifecycle status

---

> **Role:** `CORP_DQ_ADMIN` | **Time:** ~60 minutes

> **What this does:** Sets your session context to the lab role, database, and warehouse.

In [ ]:
USE ROLE CORP_DQ_ADMIN;
USE DATABASE CORP_DWH;
USE WAREHOUSE COMPUTE_WH;

---
## 3a. Create the Rules Catalog

> **Business Value:** Codifying tribal knowledge into a catalog prevents "single point of failure" when key team members leave. Rules survive people. Table

> **DQ Domain:** All (governance infrastructure)

In [ ]:
CREATE OR REPLACE TABLE CORP_DWH.DQ.RULES_CATALOG (
    RULE_ID NUMBER AUTOINCREMENT,
    RULE_NAME STRING NOT NULL,
    RULE_TYPE STRING NOT NULL COMMENT 'REGEX, RANGE, ENUM, FRESHNESS, COMPLETENESS, UNIQUENESS',
    TARGET_SCHEMA STRING NOT NULL,
    TARGET_TABLE STRING NOT NULL,
    TARGET_COLUMN STRING,
    REGEX_PATTERN STRING,
    MIN_VALUE NUMBER,
    MAX_VALUE NUMBER,
    ENUM_VALUES STRING,
    SEVERITY STRING DEFAULT 'MEDIUM' COMMENT 'LOW, MEDIUM, HIGH, CRITICAL',
    OWNER STRING,
    DMF_NAME STRING COMMENT 'Auto-generated DMF name',
    IS_ACTIVE BOOLEAN DEFAULT TRUE,
    CREATED_AT TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
    UPDATED_AT TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

---
## 3b. Seed Business Rules

> **Business Value:** Each rule represents a past incident or regulatory requirement. The catalog is the institutional memory of data quality lessons learned.

> **DQ Domain:** Accuracy, Validity, Completeness, Uniqueness, Freshness

In [ ]:
INSERT INTO CORP_DWH.DQ.RULES_CATALOG
    (RULE_NAME, RULE_TYPE, TARGET_SCHEMA, TARGET_TABLE, TARGET_COLUMN, REGEX_PATTERN, MIN_VALUE, MAX_VALUE, ENUM_VALUES, SEVERITY, OWNER)
VALUES
    ('National ID Format', 'REGEX', 'GOLD', 'DIM_CUSTOMER', 'NATIONAL_ID', '^[12][0-9]{9}$', NULL, NULL, NULL, 'CRITICAL', 'Compliance Team'),
    ('IBAN Format', 'REGEX', 'GOLD', 'DIM_CUSTOMER', 'IBAN', '^SA[0-9A-Za-z]{22}$', NULL, NULL, NULL, 'HIGH', 'Finance Team'),
    ('Phone Format', 'REGEX', 'GOLD', 'DIM_CUSTOMER', 'PHONE', '^(\\+966|05|00966)[0-9]{8,9}$', NULL, NULL, NULL, 'MEDIUM', 'CRM Team'),
    ('Email Format', 'REGEX', 'GOLD', 'DIM_CUSTOMER', 'EMAIL', '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$', NULL, NULL, NULL, 'HIGH', 'CRM Team'),
    ('Transaction Amount Range', 'RANGE', 'GOLD', 'FACT_TRANSACTIONS', 'AMOUNT', NULL, 0, 1000000, NULL, 'HIGH', 'Finance Team'),
    ('Valid Transaction Types', 'ENUM', 'GOLD', 'FACT_TRANSACTIONS', 'TXN_TYPE', NULL, NULL, NULL, 'PAYMENT,INVOICE,REFUND,TRANSFER', 'MEDIUM', 'Finance Team'),
    ('Valid Cities', 'ENUM', 'GOLD', 'DIM_CUSTOMER', 'CITY', NULL, NULL, NULL, 'Riyadh,Jeddah,Dammam,Makkah,Tabuk,Abha,Khobar,Najran', 'LOW', 'Operations'),
    ('Customer Name Not Null', 'COMPLETENESS', 'GOLD', 'DIM_CUSTOMER', 'CUSTOMER_NAME', NULL, NULL, NULL, NULL, 'CRITICAL', 'Data Governance'),
    ('National ID Uniqueness', 'UNIQUENESS', 'GOLD', 'DIM_CUSTOMER', 'NATIONAL_ID', NULL, NULL, NULL, NULL, 'CRITICAL', 'Compliance Team'),
    ('Transaction Freshness', 'FRESHNESS', 'GOLD', 'FACT_TRANSACTIONS', 'LOADED_AT', NULL, NULL, 7200, NULL, 'HIGH', 'Data Engineering'),
    ('Customer Source Valid', 'ENUM', 'GOLD', 'DIM_CUSTOMER', 'SOURCE_SYSTEM', NULL, NULL, NULL, 'ERP,CRM,HR,BANK', 'MEDIUM', 'Data Engineering'),
    ('IBAN Not Null for ERP', 'COMPLETENESS', 'GOLD', 'DIM_CUSTOMER', 'IBAN', NULL, NULL, NULL, NULL, 'HIGH', 'Finance Team');

---
## Checkpoint 1: Verify Rules Catalog

> **What this does:** Verifies your work so far. All checks should show [PASS].

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

rules = session.sql("SELECT COUNT(*) AS CNT FROM CORP_DWH.DQ.RULES_CATALOG").collect()[0]['CNT']
by_type = session.sql("""
SELECT RULE_TYPE, COUNT(*) AS CNT FROM CORP_DWH.DQ.RULES_CATALOG GROUP BY RULE_TYPE ORDER BY CNT DESC
""").to_pandas()

print("=" * 50)
print("CHECKPOINT 1: Rules Catalog Verification")
print("=" * 50)

if rules == 12:
    print(f"  [PASS] Catalog has {rules} rules (expected 12)")
else:
    print(f"  [FAIL] Catalog has {rules} rules (expected 12)")

print(f"\n  Rules by type:")
for _, row in by_type.iterrows():
    print(f"    {row['RULE_TYPE']}: {row['CNT']}")

critical = session.sql("SELECT COUNT(*) AS CNT FROM CORP_DWH.DQ.RULES_CATALOG WHERE SEVERITY = 'CRITICAL'").collect()[0]['CNT']
print(f"\n  CRITICAL severity rules: {critical}")
print("=" * 50)

---
## 3c. View the Rules Catalog

> **What this does:** Displays all rules in the catalog with their type, target, severity, and owner for a quick overview.

In [ ]:
SELECT RULE_NAME, RULE_TYPE, TARGET_TABLE, TARGET_COLUMN, SEVERITY, OWNER, IS_ACTIVE
FROM CORP_DWH.DQ.RULES_CATALOG
ORDER BY SEVERITY DESC, RULE_NAME;

---
## 3d. Build Auto-Provisioning

> **Business Value:** Manual DMF creation takes 30+ minutes per rule. Auto-provisioning reduces this to seconds, enabling stewards to add 50 rules in an afternoon. Procedure

This procedure reads the catalog and creates DMFs automatically for REGEX, RANGE, and ENUM rules.

> **DQ Domain:** All (automation)

In [ ]:
CREATE OR REPLACE PROCEDURE CORP_DWH.DQ.PROVISION_DMFS_FROM_CATALOG()
RETURNS STRING
LANGUAGE SQL
AS
DECLARE
    rules_processed NUMBER DEFAULT 0;
    dmf_sql STRING;
    attach_sql STRING;
    dmf_name STRING;
    cur CURSOR FOR
        SELECT RULE_ID, RULE_NAME, RULE_TYPE, TARGET_SCHEMA, TARGET_TABLE, TARGET_COLUMN,
               REGEX_PATTERN, MIN_VALUE, MAX_VALUE, ENUM_VALUES
        FROM CORP_DWH.DQ.RULES_CATALOG
        WHERE IS_ACTIVE = TRUE AND DMF_NAME IS NULL
          AND RULE_TYPE IN ('REGEX', 'RANGE', 'ENUM');
BEGIN
    FOR rec IN cur DO
        dmf_name := 'CORP_DWH.DQ.DMF_' || REPLACE(UPPER(rec.RULE_NAME), ' ', '_');

        IF (rec.RULE_TYPE = 'REGEX') THEN
            dmf_sql := 'CREATE OR REPLACE DATA METRIC FUNCTION ' || :dmf_name ||
                '(ARG_T TABLE(ARG_C STRING)) RETURNS NUMBER AS $$ ' ||
                'SELECT COUNT(*) FROM ARG_T WHERE ARG_C IS NOT NULL AND NOT RLIKE(ARG_C, ''' || rec.REGEX_PATTERN || ''') $$';
        ELSEIF (rec.RULE_TYPE = 'RANGE') THEN
            dmf_sql := 'CREATE OR REPLACE DATA METRIC FUNCTION ' || :dmf_name ||
                '(ARG_T TABLE(ARG_C NUMBER)) RETURNS NUMBER AS $$ ' ||
                'SELECT COUNT(*) FROM ARG_T WHERE ARG_C < ' || rec.MIN_VALUE || ' OR ARG_C > ' || rec.MAX_VALUE || ' $$';
        ELSEIF (rec.RULE_TYPE = 'ENUM') THEN
            dmf_sql := 'CREATE OR REPLACE DATA METRIC FUNCTION ' || :dmf_name ||
                '(ARG_T TABLE(ARG_C STRING)) RETURNS NUMBER AS $$ ' ||
                'SELECT COUNT(*) FROM ARG_T WHERE ARG_C IS NOT NULL AND ARG_C NOT IN (''' ||
                REPLACE(rec.ENUM_VALUES, ',', ''',''') || ''') $$';
        END IF;

        EXECUTE IMMEDIATE :dmf_sql;

        attach_sql := 'ALTER TABLE CORP_DWH.' || rec.TARGET_SCHEMA || '.' || rec.TARGET_TABLE ||
            ' ADD DATA METRIC FUNCTION ' || :dmf_name || ' ON (' || rec.TARGET_COLUMN || ')';
        EXECUTE IMMEDIATE :attach_sql;

        UPDATE CORP_DWH.DQ.RULES_CATALOG SET DMF_NAME = :dmf_name, UPDATED_AT = CURRENT_TIMESTAMP()
        WHERE RULE_ID = rec.RULE_ID;

        rules_processed := rules_processed + 1;
    END FOR;

    RETURN 'Provisioned ' || :rules_processed || ' DMFs from catalog.';
END;

---
## 3e. Run the Provisioning Procedure

> **What this does:** Executes the provisioning procedure to auto-create DMFs from all active catalog rules that haven't been provisioned yet.

In [ ]:
CALL CORP_DWH.DQ.PROVISION_DMFS_FROM_CATALOG();

---
## Checkpoint 2: Verify Provisioning Worked

> **What this does:** Verifies your work so far. All checks should show [PASS].

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

provisioned = session.sql("""
SELECT COUNT(*) AS CNT FROM CORP_DWH.DQ.RULES_CATALOG WHERE DMF_NAME IS NOT NULL
""").collect()[0]['CNT']

unprovisioned = session.sql("""
SELECT RULE_NAME, RULE_TYPE FROM CORP_DWH.DQ.RULES_CATALOG
WHERE DMF_NAME IS NULL AND IS_ACTIVE = TRUE AND RULE_TYPE IN ('REGEX','RANGE','ENUM')
""").to_pandas()

print("=" * 50)
print("CHECKPOINT 2: Auto-Provisioning Verification")
print("=" * 50)

if provisioned >= 7:
    print(f"  [PASS] {provisioned} rules have DMFs provisioned (expected >= 7)")
else:
    print(f"  [FAIL] Only {provisioned} rules provisioned (expected >= 7)")

if len(unprovisioned) == 0:
    print(f"  [PASS] No REGEX/RANGE/ENUM rules left unprovisioned")
else:
    print(f"  [WARN] {len(unprovisioned)} rules still need provisioning:")
    for _, r in unprovisioned.iterrows():
        print(f"         - {r['RULE_NAME']} ({r['RULE_TYPE']})")

# Verify DMFs exist in DQ schema
dmf_count = session.sql("""
SELECT COUNT(*) AS CNT FROM CORP_DWH.INFORMATION_SCHEMA.FUNCTIONS
WHERE FUNCTION_SCHEMA = 'DQ' AND FUNCTION_NAME LIKE 'DMF_%'
""").collect()[0]['CNT']

print(f"  [INFO] Total DMF_ functions in DQ schema: {dmf_count}")
print("=" * 50)

---
## 3f. Consistency Rules

> **Business Value:** Cross-column inconsistencies (ERP customer without IBAN) indicate broken integrations that cause payment failures and regulatory non-compliance.: Cross-Column Validation

> **DQ Domain:** Consistency | **Severity:** HIGH

Consistency rules enforce logic **across multiple columns**. These cannot be expressed as single-column DMFs -- they require multi-column table arguments.

Examples:
- If `SOURCE_SYSTEM = 'ERP'`, then `IBAN` must NOT be NULL
- If `TXN_TYPE = 'REFUND'`, then `AMOUNT` should be positive (refunds are stored as positive values in our Gold model)
- `LAST_UPDATED` must never be before `CREATED_DATE`

In [ ]:
-- Consistency: ERP customers must have an IBAN
CREATE OR REPLACE DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_ERP_IBAN_CONSISTENCY(
    ARG_T TABLE(ARG_C1 STRING, ARG_C2 STRING)
)
RETURNS NUMBER
AS
$$
    SELECT COUNT(*)
    FROM ARG_T
    WHERE ARG_C1 = 'ERP' AND ARG_C2 IS NULL
$$;

ALTER TABLE CORP_DWH.GOLD.DIM_CUSTOMER
    ADD DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_ERP_IBAN_CONSISTENCY
    ON (SOURCE_SYSTEM, IBAN);

> **What this does:** Creates a consistency DMF that flags records where LAST_UPDATED is before CREATED_DATE, detecting temporal order violations.

In [ ]:
-- Consistency: LAST_UPDATED must not be before CREATED_DATE
CREATE OR REPLACE DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_TEMPORAL_ORDER(
    ARG_T TABLE(ARG_C1 TIMESTAMP_LTZ, ARG_C2 TIMESTAMP_LTZ)
)
RETURNS NUMBER
AS
$$
    SELECT COUNT(*)
    FROM ARG_T
    WHERE ARG_C2 < ARG_C1
$$;

ALTER TABLE CORP_DWH.GOLD.DIM_CUSTOMER
    ADD DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_TEMPORAL_ORDER
    ON (CREATED_DATE, LAST_UPDATED);

---
## Checkpoint: Consistency Rules Verification

> **What this does:** Verifies your work so far. All checks should show [PASS].

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

print("=" * 50)
print("CHECKPOINT: Consistency Rules")
print("=" * 50)

# Test ERP/IBAN consistency - should find 1 (Huda has ERP source but NULL IBAN? Actually check data)
erp_no_iban = session.sql("""
SELECT COUNT(*) AS CNT FROM CORP_DWH.GOLD.DIM_CUSTOMER
WHERE SOURCE_SYSTEM = 'ERP' AND IBAN IS NULL
""").collect()[0]['CNT']
print(f"  ERP customers missing IBAN: {erp_no_iban}")
if erp_no_iban == 0:
    print(f"  [PASS] All ERP customers have IBANs")
else:
    print(f"  [FAIL] {erp_no_iban} ERP customers missing IBAN (consistency violation!)")

# Test temporal order - Reem has LAST_UPDATED 5 days ago but CREATED_DATE in 2024
temporal_issues = session.sql("""
SELECT COUNT(*) AS CNT FROM CORP_DWH.GOLD.DIM_CUSTOMER
WHERE LAST_UPDATED < CREATED_DATE
""").collect()[0]['CNT']
print(f"  Temporal order violations (LAST_UPDATED < CREATED_DATE): {temporal_issues}")
if temporal_issues == 0:
    print(f"  [PASS] All records have correct temporal order")
else:
    print(f"  [FAIL] {temporal_issues} records have LAST_UPDATED before CREATED_DATE")

print("=" * 50)

> **What this does:** Enables automatic DMF execution on Gold tables whenever data changes, ensuring continuous monitoring.

In [ ]:
-- Set schedule on Gold tables
ALTER TABLE CORP_DWH.GOLD.DIM_CUSTOMER SET DATA_METRIC_SCHEDULE = 'TRIGGER_ON_CHANGES';
ALTER TABLE CORP_DWH.GOLD.FACT_TRANSACTIONS SET DATA_METRIC_SCHEDULE = 'TRIGGER_ON_CHANGES';

---
## Quiz: Test Your Knowledge

**Q1:** The provisioning procedure skips rules with `RULE_TYPE` of 'COMPLETENESS', 'UNIQUENESS', and 'FRESHNESS'. Why?

**Q2:** If you add a new rule to the catalog and call `PROVISION_DMFS_FROM_CATALOG()` again, will it re-create existing DMFs?

**Q3:** What column in RULES_CATALOG tracks whether a rule has been provisioned?

**Q4:** A steward wants to disable a rule temporarily without deleting it. How?

---
## Drill Down: Which Gold Records Violate Business Rules?

After DMFs flag violations, identify the exact records and their catalog rule violations:

In [ ]:
-- Gold customers failing format rules (from Rules Catalog)
SELECT
    c.CUSTOMER_NAME,
    c.NATIONAL_ID,
    c.IBAN,
    c.CITY,
    c.SOURCE_SYSTEM,
    r.RULE_NAME,
    r.SEVERITY,
    r.OWNER AS RULE_OWNER,
    CASE
        WHEN r.RULE_TYPE = 'REGEX' AND NOT RLIKE(
            CASE r.TARGET_COLUMN
                WHEN 'NATIONAL_ID' THEN c.NATIONAL_ID
                WHEN 'IBAN' THEN c.IBAN
                WHEN 'PHONE' THEN c.PHONE
                WHEN 'EMAIL' THEN c.EMAIL
                ELSE ''
            END, r.REGEX_PATTERN)
        THEN 'REGEX VIOLATION: value does not match ' || r.REGEX_PATTERN
        WHEN r.RULE_TYPE = 'ENUM' AND
            CASE r.TARGET_COLUMN WHEN 'CITY' THEN c.CITY WHEN 'SOURCE_SYSTEM' THEN c.SOURCE_SYSTEM ELSE '' END
            NOT IN (SELECT VALUE FROM TABLE(SPLIT_TO_TABLE(r.ENUM_VALUES, ',')))
        THEN 'ENUM VIOLATION: value not in allowed list'
        ELSE 'PASS'
    END AS VIOLATION_DETAIL
FROM CORP_DWH.GOLD.DIM_CUSTOMER c
CROSS JOIN CORP_DWH.DQ.RULES_CATALOG r
WHERE r.IS_ACTIVE = TRUE
    AND r.TARGET_TABLE = 'DIM_CUSTOMER'
    AND r.RULE_TYPE IN ('REGEX', 'ENUM')
HAVING VIOLATION_DETAIL != 'PASS'
ORDER BY r.SEVERITY DESC, c.CUSTOMER_NAME;

> **What this does:** Reveals quiz answers. Try answering first!

In [ ]:
print("""
QUIZ ANSWERS
============

Q1: COMPLETENESS, UNIQUENESS, and FRESHNESS require different DMF signatures:
    - COMPLETENESS = NULL_COUNT (system DMF already exists)
    - UNIQUENESS = DUPLICATE_COUNT (system DMF)
    - FRESHNESS = FRESHNESS (system DMF, needs TIMESTAMP column)
    The procedure only handles REGEX/RANGE/ENUM which have a consistent
    pattern (return count of violations based on a single column check).

Q2: NO. The procedure checks 'WHERE DMF_NAME IS NULL' -- it only processes
    rules that haven't been provisioned yet. Existing rules are skipped.
    This is idempotent by design.

Q3: The DMF_NAME column. When NULL, the rule hasn't been provisioned.
    After provisioning, it stores the full DMF path (e.g., CORP_DWH.DQ.DMF_NATIONAL_ID_FORMAT).

Q4: Set IS_ACTIVE = FALSE:
    UPDATE CORP_DWH.DQ.RULES_CATALOG SET IS_ACTIVE = FALSE WHERE RULE_NAME = '...';
    The procedure checks IS_ACTIVE = TRUE, so disabled rules are skipped.
    The existing DMF stays attached but won't be re-created if dropped.
""")

---
## Challenge (Self-Guided)

1. Add a new rule: "Customer Created Date Not Future" (RANGE type, MAX_VALUE = extract of today as number)
2. Re-run `PROVISION_DMFS_FROM_CATALOG()` and verify the new DMF appears
3. Query `RULES_CATALOG` to see the updated DMF_NAME

---

**Next:** Open `4_EXPECTATIONS` to add pass/fail verdicts.